# 18. Boilerplate: does the prediction come from the lines an outlet repeats?

The anonymization leaves in the text the lines that an outlet repeats in many articles (newsletter invitations, donation
appeals, disclaimers, captions). Following the idea of the explanation methods (look at which parts of the text drive a
prediction), four checks:

1. **Boilerplate**: a line that appears, digits ignored, in at least 3 articles and 5% of the articles of the outlet.
2. **TF-IDF retrained** without boilerplate, and without boilerplate and dates (same folds and inner choice as step 7).
3. **Saved models** (GPU): on 1,000 articles with boilerplate, the predictions on the original and on the cleaned text.
4. **Mistral-24B token by token** (GPU): the prediction is linear in the mean of the token states, so it splits exactly
   into one contribution per token; the part of the boilerplate tokens.

Details in `boilerplate.py`; the numbers of the paper are in `paper_boilerplate.csv`.

In [1]:
N_JOBS = 16
GPU_GB = 19
B = 10_000
GPU = True                        # False: only the checks on the CPU (1 and 2)

In [2]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import boilerplate as BP
import metrics as ME
import models as MO
import paper as PA

art, outlets = MO.load()
art["text"] = art.text.fillna("")
y = outlets.score.to_numpy()
idx = PA.resamples(len(outlets), B)
out = []                                   # one row per check, model and measure


def add(check, model, measure, x, ref=None):
    x = np.atleast_1d(np.asarray(x, dtype=float))
    r = {"check": check, "model": model, "measure": measure, "value": x[0]}
    if len(x) > 1:
        r.update(low=np.percentile(x[1:], 2.5), high=np.percentile(x[1:], 97.5))
    if ref is not None:
        r.update(PA.compare(x, ref, measure))
    out.append(r)

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


/home/fabio.pinelli/John/johnVenv/lib/python3.12/site-packages/torch/cuda/__init__.py:716: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


## 1. How much boilerplate

In [3]:
bp = BP.find(art)
clean = BP.strip(art, bp)
for k, v in BP.summary(art, bp, clean).items():
    add("boilerplate", "", k, v)
    print(f"{k}: {v:,.3f}" if isinstance(v, float) else f"{k}: {v:,}")
pd.Series(sorted({line for _, line in bp})).sample(15, random_state=C.SEED).to_frame("examples (normalized)")

lines: 1,040
outlets: 171
outlets in all: 226
share of words: 0.030
share of articles: 0.491


,examples (normalized)
789,the run-up to the vote on tuesday is exposing the façade of israeli democracy—and the ...
497,join [outlet] vip and use promo code fight to receive 00% off your membership.
139,[author]'s note: do you enjoy [outlet]'s conservative reporting exposing the radical l...
570,note: the information provided here or in any related communications is for informatio...
66,- president trump survives another possible assassination attempt as shooter breaks up...
759,"thanks for reading! we’re an independent, reader-supported site that depends entirely ..."
799,the “mike spike”: lawler under fire for rising costs
613,please verify
967,▶ video
307,can i receive a recording of a workshop?


## 2. TF-IDF retrained without boilerplate

In [4]:
paper = MO.predictions("regression", "TF-IDF").groupby("domain").prediction.mean().reindex(outlets.index).to_numpy()
ref = ME.resampled_measures(paper, y, idx, ["MAE", "Spearman"])
for mes, x in ref.items():
    add("TF-IDF retrained", "as in the paper", mes, x)
for version, texts in [("clean", clean), ("nodates", [BP.DATE.sub(" [DATE] ", t) for t in clean])]:
    p = BP.retrain_tfidf(art, texts, version, N_JOBS, LOG).reindex(outlets.index).to_numpy()
    for mes, x in ME.resampled_measures(p, y, idx, ["MAE", "Spearman"]).items():
        add("TF-IDF retrained", "without boilerplate" + (" and dates" if version == "nodates" else ""), mes, x, ref[mes])
pd.DataFrame(out).to_csv(C.OUT / "paper_boilerplate.csv", index=False)      # saved now: the GPU may be busy
pd.DataFrame(out).query("check == 'TF-IDF retrained'").round(3)

2026-11-23 11:01:49  regression, TF-IDF, fold 1: penalty 0.1 (inside the grid), 120 s


2026-11-23 11:03:49  regression, TF-IDF, fold 2: penalty 0.003 (inside the grid), 119 s


2026-11-23 11:05:46  regression, TF-IDF, fold 3: penalty 0.001 (on the grid edge), 117 s


2026-11-23 11:07:36  regression, TF-IDF, fold 4: penalty 0.03 (inside the grid), 109 s


2026-11-23 11:09:24  regression, TF-IDF, fold 5: penalty 0.3 (inside the grid), 109 s


2026-11-23 11:11:20  regression, TF-IDF, fold 1: penalty 0.03 (inside the grid), 116 s


2026-11-23 11:13:19  regression, TF-IDF, fold 2: penalty 0.01 (inside the grid), 119 s


2026-11-23 11:15:17  regression, TF-IDF, fold 3: penalty 0.001 (on the grid edge), 118 s


2026-11-23 11:17:08  regression, TF-IDF, fold 4: penalty 0.03 (inside the grid), 110 s


2026-11-23 11:19:03  regression, TF-IDF, fold 5: penalty 0.01 (inside the grid), 116 s


,check,model,measure,value,low,high,diff,dlo,dhi,p
5,TF-IDF retrained,as in the paper,Spearman,0.674,0.604,0.733,NaN,NaN,NaN,NaN
6,TF-IDF retrained,as in the paper,MAE,17.825,16.187,19.553,NaN,NaN,NaN,NaN
7,TF-IDF retrained,without boilerplate,Spearman,0.677,0.605,0.736,0.002,-0.029,0.034,0.868
8,TF-IDF retrained,without boilerplate,MAE,17.501,15.845,19.237,-0.324,-0.737,0.104,0.133
9,TF-IDF retrained,without boilerplate and dates,Spearman,0.683,0.613,0.741,0.009,-0.023,0.041,0.547
10,TF-IDF retrained,without boilerplate and dates,MAE,17.371,15.730,19.079,-0.454,-0.869,-0.031,0.036


## 3. The saved models on 1,000 articles with boilerplate (GPU)

In [5]:
if GPU:
    S = BP.sample(art, clean)
    P = BP.sample_predictions(S, GPU_GB, LOG)
    add("sample", "", "articles", len(P))
    add("sample", "", "share of words", P.share.mean())
    for m in ["TF-IDF", "Qwen3-8B", "Mistral-24B"]:
        o, c = P[f"{m} original"], P[f"{m} clean"]
        add("sample", m, "MAE original", (o - P.score).abs().mean())
        add("sample", m, "MAE without boilerplate", (c - P.score).abs().mean())
        add("sample", m, "mean absolute change", (c - o).abs().mean())
        add("sample", m, "mean change", (c - o).mean())
        add("sample", m, "Pearson r", np.corrcoef(o, c)[0, 1])
    display(pd.DataFrame(out).query("check == 'sample'").pivot(index="measure", columns="model", values="value").round(3))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards:  25%|██▌       | 1/4 [00:02<00:06,  2.01s/it]

Loading checkpoint shards:  50%|█████     | 2/4 [00:03<00:03,  1.95s/it]

Loading checkpoint shards:  75%|███████▌  | 3/4 [00:05<00:01,  1.92s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.22s/it]

Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.48s/it]

... (12 outputs left out)

Loading checkpoint shards: 100%|██████████| 10/10 [00:53<00:00,  5.38s/it]

2026-11-23 11:24:01  Mistral: 2/1,000 articles


2026-11-23 11:31:33  Mistral: 664/1,000 articles


model,,Mistral-24B,Qwen3-8B,TF-IDF
measure,,,,
MAE original,NaN,17.016,18.711,19.335
MAE without boilerplate,NaN,17.533,19.501,20.421
Pearson r,NaN,0.982,0.936,0.954
articles,1000.000,NaN,NaN,NaN
mean absolute change,NaN,2.040,4.823,3.311
mean change,NaN,-0.724,-1.488,-1.143
share of words,0.087,NaN,NaN,NaN


## 4. Mistral-24B token by token (GPU)

In [6]:
if GPU:
    T = BP.mistral_tokens(S, bp, 300, GPU_GB, LOG)
    part, pred = T["boilerplate part"], T.prediction
    add("tokens", "Mistral-24B", "articles", len(T))
    add("tokens", "Mistral-24B", "max |rebuilt - saved prediction|", (pred - T.saved).abs().max())
    add("tokens", "Mistral-24B", "share of boilerplate tokens", T["boilerplate tokens"].mean())
    add("tokens", "Mistral-24B", "mean |boilerplate part|", part.abs().mean())
    add("tokens", "Mistral-24B", "sd of the predictions", pred.std())
    add("tokens", "Mistral-24B", "share of the variance (covariance)", np.cov(part, pred)[0, 1] / pred.var())
    add("tokens", "Mistral-24B", "variance of the part / variance", part.var() / pred.var())
    display(pd.DataFrame(out).query("check == 'tokens'").round(3))

Loading checkpoint shards:   0%|          | 0/10 [00:00<?, ?it/s]

Loading checkpoint shards:  10%|█         | 1/10 [00:05<00:51,  5.72s/it]

Loading checkpoint shards:  20%|██        | 2/10 [00:12<00:49,  6.22s/it]

Loading checkpoint shards:  30%|███       | 3/10 [00:18<00:44,  6.36s/it]

Loading checkpoint shards:  40%|████      | 4/10 [00:25<00:38,  6.49s/it]

Loading checkpoint shards:  50%|█████     | 5/10 [00:32<00:32,  6.52s/it]

... (4 outputs left out)

Loading checkpoint shards: 100%|██████████| 10/10 [01:02<00:00,  5.84s/it]

Loading checkpoint shards: 100%|██████████| 10/10 [01:02<00:00,  6.28s/it]

2026-11-23 11:37:51  boilerplate: Mistral token by token on 300 articles


,check,model,measure,value,low,high,diff,dlo,dhi,p
28,tokens,Mistral-24B,articles,300.000,NaN,NaN,NaN,NaN,NaN,NaN
29,tokens,Mistral-24B,max |rebuilt - saved prediction|,0.041,NaN,NaN,NaN,NaN,NaN,NaN
30,tokens,Mistral-24B,share of boilerplate tokens,0.082,NaN,NaN,NaN,NaN,NaN,NaN
31,tokens,Mistral-24B,mean |boilerplate part|,1.950,NaN,NaN,NaN,NaN,NaN,NaN
32,tokens,Mistral-24B,sd of the predictions,17.462,NaN,NaN,NaN,NaN,NaN,NaN
33,tokens,Mistral-24B,share of the variance (covariance),0.118,NaN,NaN,NaN,NaN,NaN,NaN
34,tokens,Mistral-24B,variance of the part / variance,0.035,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
R = pd.DataFrame(out)
R.to_csv(C.OUT / "paper_boilerplate.csv", index=False)
R.round(3)

,check,model,measure,value,low,high,diff,dlo,dhi,p
0,boilerplate,,lines,1040.000,NaN,NaN,NaN,NaN,NaN,NaN
1,boilerplate,,outlets,171.000,NaN,NaN,NaN,NaN,NaN,NaN
2,boilerplate,,outlets in all,226.000,NaN,NaN,NaN,NaN,NaN,NaN
3,boilerplate,,share of words,0.030,NaN,NaN,NaN,NaN,NaN,NaN
4,boilerplate,,share of articles,0.491,NaN,NaN,NaN,NaN,NaN,NaN
5,TF-IDF retrained,as in the paper,Spearman,0.674,0.604,0.733,NaN,NaN,NaN,NaN
6,TF-IDF retrained,as in the paper,MAE,17.825,16.187,19.553,NaN,NaN,NaN,NaN
7,TF-IDF retrained,without boilerplate,Spearman,0.677,0.605,0.736,0.002,-0.029,0.034,0.868
8,TF-IDF retrained,without boilerplate,MAE,17.501,15.845,19.237,-0.324,-0.737,0.104,0.133
9,TF-IDF retrained,without boilerplate and dates,Spearman,0.683,0.613,0.741,0.009,-0.023,0.041,0.547
